#Task 2: Calculate Scale and Zero Point


##Objective

Calculate the quantization parameters (scale and zero point) required to map a floating-point tensor to the INT8 range. In Task 1, these were given; now you compute them yourself.

##Formulas

q_min = -128,  q_max = 127

scale = (x_max - x_min) / (q_max - q_min)

zero_point = round(q_min - (x_min / scale))

zero_point = clip(zero_point, q_min, q_max)



In [1]:
#import the req libs

import numpy as np

In [3]:
# Tensor 1: Mixed positive and negative values
t1 = np.array([-1.5, -0.8, 0.0, 0.9, 2.3], dtype=np.float32)

# Tensor 2: All positive values
t2 = np.array([0.1, 0.5, 1.2, 2.0, 3.5], dtype=np.float32)

# Tensor 3: All negative values
t3 = np.array([-3.0, -2.1, -1.4, -0.6, -0.1], dtype=np.float32)

# Tensor 4: Constant tensor
t4 = np.array([5.0, 5.0, 5.0], dtype=np.float32)

# Tensor 5: Very small values
t5 = np.array([1e-9, 2e-9, -1e-9], dtype=np.float32)

In [4]:
def calculate_scale_zero_point(tensor, q_min=-128, q_max=127):

    # Calculate scale and zero point for affine INT8 quantization
    # Handle edge cases: constant tensor, very small range
    # Return: (scale, zero_point)

    x_min = np.min(tensor)
    x_max = np.max(tensor)

    #handle case where the tensor is constant to avoid div by zero error
    if x_max == x_min:
      return 1.0,0

    scale = (x_max - x_min) / (q_max - q_min)
    zero_point = np.round(q_min - (x_min/scale))
    zero_point = np.clip(zero_point,q_min,q_max)

    return scale,int(zero_point)

In [5]:
#using the same fucn from task1 for quant and dequant
def quantize_tensor(tensor, scale, zero_point):
    quantized = tensor / scale
    quantized = np.round(quantized)
    quantized = quantized + zero_point
    quantized = np.clip(quantized, -128, 127)
    return quantized.astype(np.int8)


def dequantize_tensor(quantized_tensor, scale, zero_point):
    dequantized = quantized_tensor.astype(np.float32) - zero_point
    dequantized = dequantized * scale
    return dequantized

In [14]:
#function to execute the process

def process_tensor(name,tensor):
  print(name,":")

  scale,zero_point = calculate_scale_zero_point(tensor)

  quantized_tensor = quantize_tensor(tensor,scale,zero_point)
  dequantized_tensor = dequantize_tensor(quantized_tensor,scale,zero_point)

  mae = np.mean(np.abs(tensor - dequantized_tensor))


  print("Original Tensor:",tensor)
  print("Tensor min:",np.min(tensor))
  print("Tensor max:",np.max(tensor))
  print("Scale:",scale)
  print("Zero point:",zero_point)
  print("Quantized Tensor (INT8):",quantized_tensor)
  print("Dequantized Tensor:",dequantized_tensor)
  print("Mean Absolute Error (MAE):",mae)
  print()
  print("-------------------------------------------------------------------------")
  print()

In [15]:
process_tensor("Tensor 1 - Mixed Values", t1)
process_tensor("Tensor 2 - All Positive", t2)
process_tensor("Tensor 3 - All Negative", t3)
process_tensor("Tensor 4 - Constant Tensor", t4)
process_tensor("Tensor 5 - Very Small Values", t5)

Tensor 1 - Mixed Values :
Original Tensor: [-1.5 -0.8  0.   0.9  2.3]
Tensor min: -1.5
Tensor max: 2.3
Scale: 0.01490196
Zero point: -27
Quantized Tensor (INT8): [-128  -81  -27   33  127]
Dequantized Tensor: [-1.505098   -0.80470586  0.          0.8941176   2.2949018 ]
Mean Absolute Error (MAE): 0.004156864

-------------------------------------------------------------------------

Tensor 2 - All Positive :
Original Tensor: [0.1 0.5 1.2 2.  3.5]
Tensor min: 0.1
Tensor max: 3.5
Scale: 0.013333334
Zero point: -128
Quantized Tensor (INT8): [-120  -90  -38   22  127]
Dequantized Tensor: [0.10666667 0.50666666 1.2        2.         3.4       ]
Mean Absolute Error (MAE): 0.022666646

-------------------------------------------------------------------------

Tensor 3 - All Negative :
Original Tensor: [-3.  -2.1 -1.4 -0.6 -0.1]
Tensor min: -3.0
Tensor max: -0.1
Scale: 0.011372549
Zero point: 127
Quantized Tensor (INT8): [-128  -58    4   74  118]
Dequantized Tensor: [-2.9        -2.1039217  -

## Observations

- The scale and zero point were successfully calculated for all five tensors.
- The mixed, all-positive, and all-negative tensors were quantized and dequantized with low reconstruction error.
- The constant tensor was handled correctly by avoiding division by zero using a default scale(1.0) and zero point.
- The tensor with very small values also produced valid quantization parameters and retained good precision as the scale was small enough for the given range.
